# Testes usando intervalos RR

In [35]:
import pandas as pd
df = pd.read_csv('../database/processed/Dados_Arritmia_Renan.csv')
colunas_rrs = ['rr_pre', 'rr_pos', 'rr_pre_relativo', 'rr_pos_relativo']

In [36]:
df = df[df['padrao_aami'] != 'Q'].copy()
df['padrao_aami'].value_counts()

padrao_aami
N    90069
V     7480
S     2779
F      802
Name: count, dtype: int64

In [37]:
df_treino = df[df['tipo'] == 'treino'].copy()
df_teste = df[df['tipo'] == 'teste'].copy()

In [38]:
# Separar treino/teste - Problema binario
label_mapping = {"F": 1, "N": 0, "S": 1, "V": 1}

rrs_treino = df_treino[colunas_rrs]
y_treino = df_treino['padrao_aami'].map(label_mapping).astype(int)
rrs_teste = df_teste[colunas_rrs]
y_teste = df_teste['padrao_aami'].map(label_mapping).astype(int)

In [39]:
# Testar em um XGBoost binário com penalização para a classe minoritária
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

quantidade_negativos = (y_treino == 0).sum()
quantidade_positivos = (y_treino == 1).sum()
peso_classe_positiva = quantidade_negativos / quantidade_positivos

cls = xgb.XGBClassifier(
    random_state=14,
    scale_pos_weight=peso_classe_positiva
)
cls.fit(rrs_treino, y_treino)

print(f'Peso aplicado à classe 1: {peso_classe_positiva:.4f}')


Peso aplicado à classe 1: 8.1604


In [40]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = cls.predict(rrs_teste)
matriz_confusao = confusion_matrix(y_teste, y_pred)
print("Matriz de Confusão:\n", matriz_confusao)

relatorio = classification_report(y_teste, y_pred)
print("\nRelatório de Classificação:\n", relatorio)

Matriz de Confusão:
 [[38992  5240]
 [ 1872  3572]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.95      0.88      0.92     44232
           1       0.41      0.66      0.50      5444

    accuracy                           0.86     49676
   macro avg       0.68      0.77      0.71     49676
weighted avg       0.89      0.86      0.87     49676



In [53]:
import optuna
import xgboost as xgb
from sklearn.model_selection import cross_val_score

def objective(trial):
    # Hiperparâmetros recomendados para XGBoost
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'objective': 'binary:logistic',
        'eval_metric': 'mlogloss',
        'n_jobs': -1,
        'random_state': 14
    }

    # Inicializa o classificador
    clf = xgb.XGBClassifier(**params, scale_pos_weight=peso_classe_positiva)

    # Cross-validation
    score = cross_val_score(clf, rrs_treino, y_treino.values.ravel(), scoring='recall', cv=5).mean()
    return score

# Criando e executando
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Melhor pontuação: {study.best_value:.4f}")
print("Melhores parâmetros:")
print(study.best_params)

# Treinando o modelo final com os melhores parâmetros encontrados
best_xgb = xgb.XGBClassifier(**study.best_params, scale_pos_weight=peso_classe_positiva, objective='binary:logistic', n_jobs=-1, random_state=14)
best_xgb.fit(rrs_treino, y_treino.values.ravel())

[I 2026-08-26 15:00:49,485] A new study created in memory with name: no-name-36d16394-e198-4681-9320-186c82c0802f
Best trial: 0. Best value: 0.886599:   2%|▏         | 1/50 [00:02<01:46,  2.17s/it]

[I 2026-08-26 15:00:51,659] Trial 0 finished with value: 0.8865990309383547 and parameters: {'n_estimators': 192, 'max_depth': 3, 'learning_rate': 0.06999580119375265, 'subsample': 0.6103011678781215, 'colsample_bytree': 0.6737179911506609, 'gamma': 0.1736904283863583}. Best is trial 0 with value: 0.8865990309383547.


Best trial: 1. Best value: 0.88909:   4%|▍         | 2/50 [00:03<01:20,  1.68s/it] 

[I 2026-08-26 15:00:52,985] Trial 1 finished with value: 0.8890904510351341 and parameters: {'n_estimators': 89, 'max_depth': 4, 'learning_rate': 0.09927497939290272, 'subsample': 0.9264174878490732, 'colsample_bytree': 0.7907766648578457, 'gamma': 0.9336752352770589}. Best is trial 1 with value: 0.8890904510351341.


Best trial: 1. Best value: 0.88909:   6%|▌         | 3/50 [00:10<03:19,  4.24s/it]

...

Best trial: 12. Best value: 0.89675:  98%|█████████▊| 49/50 [02:05<00:03,  3.28s/it]

[I 2026-08-26 15:02:55,235] Trial 48 finished with value: 0.857931221340905 and parameters: {'n_estimators': 105, 'max_depth': 9, 'learning_rate': 0.017330005309282477, 'subsample': 0.7277585738739014, 'colsample_bytree': 0.8630485726126125, 'gamma': 1.5763317239219083}. Best is trial 12 with value: 0.8967502527229112.


Best trial: 12. Best value: 0.89675: 100%|██████████| 50/50 [02:08<00:00,  2.57s/it]


[I 2026-08-26 15:02:58,030] Trial 49 finished with value: 0.8855291970224647 and parameters: {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.10723992743547223, 'subsample': 0.998221494933376, 'colsample_bytree': 0.7497386085203706, 'gamma': 1.286212480381356}. Best is trial 12 with value: 0.8967502527229112.
Melhor pontuação: 0.8968
Melhores parâmetros:
{'n_estimators': 109, 'max_depth': 3, 'learning_rate': 0.10469261395491605, 'subsample': 0.6815894086655203, 'colsample_bytree': 0.8765135909858891, 'gamma': 1.746601709247622}


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8765135909858891
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [54]:
y_pred_best = best_xgb.predict(rrs_teste)
matriz_confusao = confusion_matrix(y_teste, y_pred_best)
print("Matriz de Confusão:\n", matriz_confusao)

relatorio = classification_report(y_teste, y_pred_best)
print("\nRelatório de Classificação:\n", relatorio)

Matriz de Confusão:
 [[39203  5029]
 [ 1686  3758]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.96      0.89      0.92     44232
           1       0.43      0.69      0.53      5444

    accuracy                           0.86     49676
   macro avg       0.69      0.79      0.72     49676
weighted avg       0.90      0.86      0.88     49676



In [56]:
import matplotlib.pyplot as plt

# Análise de erro por classe original, incluindo Normal
df_erros = df_teste.copy()
df_erros['y_real'] = y_teste.values
df_erros['y_pred'] = y_pred_best
df_erros['acertou'] = df_erros['y_real'] == df_erros['y_pred']

# Agora considerando todas as classes originais (N, F, S, V)
resumo_erros = df_erros.groupby('padrao_aami')['acertou'].agg(
    total='count',
    acertos='sum'
)
resumo_erros['erros'] = resumo_erros['total'] - resumo_erros['acertos']
resumo_erros['taxa_erro'] = (resumo_erros['erros'] / resumo_erros['total'] * 100).round(2)
resumo_erros = resumo_erros.sort_values('taxa_erro', ascending=False)

print(resumo_erros)

             total  acertos  erros  taxa_erro
padrao_aami                                  
F              388       11    377      97.16
S             1836      730   1106      60.24
N            44232    39203   5029      11.37
V             3220     3017    203       6.30
